In [1]:
import os

In [2]:
def part1_parse(data):
    version_sum = 0
    def packet_parse(packet):
        nonlocal version_sum
        version = int(packet[0:3], 2)
        version_sum += version
        type_id = int(packet[3:6], 2)
        if type_id == 4:
            count = 0
            while True:
                offset = count * 5
                head = int(packet[6 + offset])
                count += 1
                if not head:
                    return packet[11 + offset:]
        else:
            length_type_id = int(packet[6])
            if length_type_id:
                sub_packet_num = int(packet[7:18], 2)
                packet = packet[18:]
                for x in range(sub_packet_num):
                    packet = packet_parse(packet)
                return packet
            else:
                sub_packet_len = int(packet[7:22], 2)
                sub_packet = packet[22: 22 + sub_packet_len]
                while sub_packet:
                    sub_packet = packet_parse(sub_packet)
                return packet[22 + sub_packet_len:]
    while True:
        data = packet_parse(data)
        if not data or set(data) == {'0'}:
            return version_sum

In [3]:
def part2_parse(data):
    def operation(num, operands):
        match num:
            case 0:
                return sum(operands)
            case 1:
                product = 1
                for operand in operands:
                    product *= operand
                return product
            case 2:
                return min(operands)
            case 3:
                return max(operands)
            case _:
                a, b = operands
                match num:
                    case 5:
                        return 1 if a > b else 0
                    case 6:
                        return 1 if a < b else 0
                    case 7:
                        return 1 if a == b else 0
    def packet_parse(packet):
        type_id = int(packet[3:6], 2)
        length_type_id = int(packet[6])
        if type_id == 4:
            count, bin_string = 0, ''
            while True:
                offset = count * 5
                header_bit = int(packet[6 + offset])
                bin_string += packet[7 + offset:11 + offset]
                count += 1
                if not header_bit:
                    value = int(bin_string, 2)
                    return (packet[11 + offset:], value)
        elif length_type_id:
            sub_packet_num = int(packet[7:18], 2)
            packet = packet[18:]
            operands = []
            for x in range(sub_packet_num):
                packet, operand = packet_parse(packet)
                operands.append(operand)
            value = operation(type_id, operands)
            return (packet, value)
        else:
            sub_packet_len = int(packet[7:22], 2)
            sub_packet = packet[22: 22 + sub_packet_len]
            operands = []
            while sub_packet:
                sub_packet, operand = packet_parse(sub_packet)
                operands.append(operand)
            value = operation(type_id, operands)
            return (packet[22 + sub_packet_len:], value)
    return packet_parse(data)[1]
        

In [4]:
file = 'data/day16.txt'
path = os.path.join(os.getcwd(), file)
with open(path, 'r') as fp:
    message = fp.readline()
binary_data = ''.join([format(int(x, 16), '04b') for x in message])

In [5]:
# part one
print(part1_parse(binary_data))

996


In [6]:
# part two
print(part2_parse(binary_data))

96257984154
